# Crownlands Siege Balance Test

## tl;dr

The two-layer siege direction is sound, but the current numbers need two changes before production: smooth the winner-survivor formula at the capture threshold and soften wall growth above Level 100. The 5% persistent-damage threshold, 30-minute repair window, and 8%/10% objective defense bonuses behave as designed.

## Context & Methods

This experiment evaluates the server-authoritative siege rules implemented in `functions/index.js` using the current values in `functions/economy-config.json`. The decision is whether the update is numerically ready for production and which narrow adjustments are justified.

### Key Assumptions

- Fresh, full-integrity walls unless a sequential-wave test says otherwise.
- Ordinary PvP combat with no weaker-kingdom protection or protected raid.
- Max Swordmastery is +60%; max Stoneworks is +75%.
- Baseline defender population is 1,000,000 raw troops.
- No live battle telemetry is available, so this validates mathematical behavior rather than observed player outcomes.

In [1]:
from pathlib import Path
import json
import math

ROOT = Path.cwd()
economy = json.loads((ROOT / 'functions' / 'economy-config.json').read_text(encoding='utf-8'))
server_source = (ROOT / 'functions' / 'index.js').read_text(encoding='utf-8')

BASE_ATTACK = 2.0
SWORD_MAX = economy['skills']['swordmastery']['maxPercent']
STONE_MAX = economy['skills']['stoneworks']['maxPercent']
DEFENSE_PER_LEVEL = economy['cityEconomy']['defensePercentPerLevel']
WALL_BASE = economy['cityEconomy']['wallDefenseBase']
WALL_EXPONENT = economy['cityEconomy']['wallDefenseExponent']
WALL_SCALE = economy['cityEconomy']['wallDefenseScale']
MEANINGFUL_DAMAGE = economy['siegeCombat']['meaningfulWallDamagePercent'] / 100
WALL_LOSS_CAP = economy['siegeCombat']['intactWallDefenderLossCapPercent'] / 100
REPAIR_MINUTES = economy['siegeCombat']['repairWindowMinutes']

assert 'const COMBAT_FORECAST_VERSION = 2' in server_source
assert 'penetratingAttackPower > garrisonDefensePower' in server_source
assert 'attackPower - defensePower * 0.68' in server_source
print(json.dumps({'sword_max_percent': SWORD_MAX, 'stone_max_percent': STONE_MAX, 'repair_minutes': REPAIR_MINUTES, 'meaningful_damage_percent': MEANINGFUL_DAMAGE * 100}, indent=2))

{
  "sword_max_percent": 60,
  "stone_max_percent": 75,
  "repair_minutes": 30,
  "meaningful_damage_percent": 5.0
}


In [2]:
def base_wall(level):
    return math.floor(WALL_BASE + max(0, (level ** WALL_EXPONENT - 1) * WALL_SCALE))

def full_wall(level, stone_percent=STONE_MAX, defense_bonus_percent=0, soft_cap=False):
    if soft_cap and level > 100:
        level_100_wall = base_wall(100)
        level_100_slope = WALL_SCALE * WALL_EXPONENT * (100 ** (WALL_EXPONENT - 1))
        raw = math.floor(level_100_wall + level_100_slope * (level - 100))
    else:
        raw = base_wall(level)
    stone_wall = math.floor(raw * (1 + stone_percent / 100))
    return math.floor(stone_wall * (1 + defense_bonus_percent / 100))

def garrison_power(level, defenders, defense_bonus_percent=0):
    city_power = math.floor(defenders * (1 + level * DEFENSE_PER_LEVEL / 100))
    return math.floor(city_power * (1 + defense_bonus_percent / 100))

def attack_power(troops, sword_percent=SWORD_MAX):
    return math.floor(troops * BASE_ATTACK * (1 + sword_percent / 100))

def minimum_attackers_for_power(required_power, sword_percent=SWORD_MAX):
    troops = max(1, math.floor(required_power / (BASE_ATTACK * (1 + sword_percent / 100))))
    while attack_power(troops, sword_percent) <= required_power:
        troops += 1
    return troops

def combat(attackers, defenders, level, integrity_bps=10_000, stone_percent=STONE_MAX, defense_bonus_percent=0, soft_cap=False, smooth_survivors=False):
    full = full_wall(level, stone_percent, defense_bonus_percent, soft_cap)
    wall = math.floor(full * integrity_bps / 10_000)
    garrison = garrison_power(level, defenders, defense_bonus_percent)
    attack = attack_power(attackers)
    wall_damage = min(attack, wall)
    penetrating = max(0, attack - wall)
    breached = attack >= wall
    won = penetrating > garrison
    raw_ending_bps = max(0, integrity_bps - math.ceil(wall_damage * 10_000 / full)) if full else integrity_bps
    meaningful = wall > 0 and (wall_damage >= full * MEANINGFUL_DAMAGE or raw_ending_bps <= 0)
    ending_bps = raw_ending_bps if meaningful else integrity_bps
    survivors = 0
    if won:
        if smooth_survivors:
            leftover = attack - (wall + garrison)
        else:
            leftover = attack - (wall + garrison) * 0.68
        survivors = max(1, min(attackers, math.floor(attackers * leftover / max(attack, 1))))
        defender_losses = defenders
    elif not breached:
        pressure = min(1, attack / max(1, wall))
        defender_losses = min(defenders, math.floor(defenders * min(WALL_LOSS_CAP, pressure * WALL_LOSS_CAP)))
    else:
        pressure = min(1, penetrating / max(1, garrison))
        defender_losses = min(defenders, math.floor(defenders * min(0.82, pressure * 0.82)))
    return {
        'won': won, 'attackers': attackers, 'attack_power': attack, 'survivors': survivors,
        'attacker_losses': attackers - survivors, 'defenders': defenders,
        'defender_losses': defender_losses, 'defenders_left': max(0, defenders - defender_losses),
        'full_wall': full, 'starting_wall': wall, 'garrison_power': garrison,
        'total_defense': wall + garrison, 'penetrating_power': penetrating,
        'breached': breached, 'ending_integrity_bps': ending_bps, 'meaningful_damage': meaningful,
    }

## Data

The test matrix covers Levels 1–200, equal-force attacks, fresh one-wave capture thresholds, sequential attacks inside the repair window, the one-troop capture boundary, and player-owned Ironwatch/Citadel defenses.

In [3]:
levels = [1, 25, 50, 75, 100, 125, 150, 175, 200]
fresh_matrix = []
for level in levels:
    wall = full_wall(level)
    garrison = garrison_power(level, 1_000_000)
    total = wall + garrison
    minimum_attackers = minimum_attackers_for_power(total)
    meaningful_attackers = math.ceil(wall * MEANINGFUL_DAMAGE / (BASE_ATTACK * (1 + SWORD_MAX / 100)))
    fresh_matrix.append({
        'level': level, 'wall_power': wall, 'garrison_power': garrison, 'total_defense': total,
        'wall_share': wall / total, 'minimum_attackers': minimum_attackers,
        'attacker_to_defender_ratio': minimum_attackers / 1_000_000,
        'minimum_meaningful_attackers': meaningful_attackers,
    })
print(json.dumps(fresh_matrix, indent=2))

[
  {
    "level": 1,
    "wall_power": 350,
    "garrison_power": 1020000,
    "total_defense": 1020350,
    "wall_share": 0.00034301955211447053,
    "minimum_attackers": 318860,
    "attacker_to_defender_ratio": 0.31886,
    "minimum_meaningful_attackers": 6
  },
  {
    "level": 25,
    "wall_power": 82376,
    "garrison_power": 1500000,
    "total_defense": 1582376,
    "wall_share": 0.05205842353524068,
    "minimum_attackers": 494493,
    "attacker_to_defender_ratio": 0.494493,
    "minimum_meaningful_attackers": 1288
  },
  {
    "level": 50,
    "wall_power": 656594,
    "garrison_power": 2000000,
    "total_defense": 2656594,
    "wall_share": 0.24715632121430675,
    "minimum_attackers": 830186,
    "attacker_to_defender_ratio": 0.830186,
    "minimum_meaningful_attackers": 10260
  },
  {
    "level": 75,
    "wall_power": 2215188,
    "garrison_power": 2500000,
    "total_defense": 4715188,
    "wall_share": 0.4697984470608595,
    "minimum_attackers": 1473497,
    "attacke

In [4]:
def sequential_equal_waves(level, initial_defenders=1_000_000, wave_size=1_000_000, max_waves=20, soft_cap=False):
    defenders = initial_defenders
    integrity = 10_000
    total_sent = 0
    total_lost = 0
    history = []
    for wave in range(1, max_waves + 1):
        result = combat(wave_size, defenders, level, integrity, soft_cap=soft_cap)
        total_sent += wave_size
        total_lost += result['attacker_losses']
        history.append({'wave': wave, 'won': result['won'], 'defenders_left': result['defenders_left'], 'wall_integrity_percent': result['ending_integrity_bps'] / 100, 'attacker_survivors': result['survivors']})
        if result['won']:
            return {'level': level, 'waves': wave, 'total_sent': total_sent, 'total_lost': total_lost, 'history': history}
        defenders = result['defenders_left']
        integrity = result['ending_integrity_bps']
    return {'level': level, 'waves': None, 'total_sent': total_sent, 'total_lost': total_lost, 'history': history}

wave_matrix = [sequential_equal_waves(level) for level in levels]
soft_cap_wave_matrix = [sequential_equal_waves(level, soft_cap=True) for level in levels]
print(json.dumps({'current': wave_matrix, 'soft_cap_after_100': soft_cap_wave_matrix}, indent=2))

{
  "current": [
    {
      "level": 1,
      "waves": 1,
      "total_sent": 1000000,
      "total_lost": 216825,
      "history": [
        {
          "wave": 1,
          "won": true,
          "defenders_left": 0,
          "wall_integrity_percent": 0.0,
          "attacker_survivors": 783175
        }
      ]
    },
    {
      "level": 25,
      "waves": 1,
      "total_sent": 1000000,
      "total_lost": 336255,
      "history": [
        {
          "wave": 1,
          "won": true,
          "defenders_left": 0,
          "wall_integrity_percent": 0.0,
          "attacker_survivors": 663745
        }
      ]
    },
    {
      "level": 50,
      "waves": 1,
      "total_sent": 1000000,
      "total_lost": 564527,
      "history": [
        {
          "wave": 1,
          "won": true,
          "defenders_left": 0,
          "wall_integrity_percent": 0.0,
          "attacker_survivors": 435473
        }
      ]
    },
    {
      "level": 75,
      "waves": 2,
      "total_s

In [5]:
cliff_matrix = []
for level in [50, 75, 100, 125, 150, 200]:
    total = full_wall(level) + garrison_power(level, 1_000_000)
    threshold = minimum_attackers_for_power(total)
    below = combat(threshold - 1, 1_000_000, level)
    above = combat(threshold, 1_000_000, level)
    smooth = combat(threshold, 1_000_000, level, smooth_survivors=True)
    cliff_matrix.append({
        'level': level, 'threshold_attackers': threshold,
        'losses_one_troop_below': below['attacker_losses'],
        'current_survivors_at_threshold': above['survivors'],
        'smoothed_survivors_at_threshold': smooth['survivors'],
    })
print(json.dumps(cliff_matrix, indent=2))

[
  {
    "level": 50,
    "threshold_attackers": 830186,
    "losses_one_troop_below": 830185,
    "current_survivors_at_threshold": 265659,
    "smoothed_survivors_at_threshold": 1
  },
  {
    "level": 75,
    "threshold_attackers": 1473497,
    "losses_one_troop_below": 1473496,
    "current_survivors_at_threshold": 471519,
    "smoothed_survivors_at_threshold": 1
  },
  {
    "level": 100,
    "threshold_attackers": 2578233,
    "losses_one_troop_below": 2578232,
    "current_survivors_at_threshold": 825034,
    "smoothed_survivors_at_threshold": 1
  },
  {
    "level": 125,
    "threshold_attackers": 4298204,
    "losses_one_troop_below": 4298203,
    "current_survivors_at_threshold": 1375425,
    "smoothed_survivors_at_threshold": 1
  },
  {
    "level": 150,
    "threshold_attackers": 6787218,
    "losses_one_troop_below": 6787217,
    "current_survivors_at_threshold": 2171910,
    "smoothed_survivors_at_threshold": 1
  },
  {
    "level": 200,
    "threshold_attackers": 146876

In [6]:
objective_matrix = []
for name, level, defenders, bonus in [
    ('Owned Level 50 stronghold without defense objective', 50, 50_000_000, 0),
    ('Owned Ironwatch', 50, 50_000_000, 8),
    ('Owned Crown Citadel', 100, 50_000_000, 10),
]:
    wall = full_wall(level, defense_bonus_percent=bonus)
    garrison = garrison_power(level, defenders, bonus)
    threshold = minimum_attackers_for_power(wall + garrison)
    objective_matrix.append({'target': name, 'defense_bonus_percent': bonus, 'wall_power': wall, 'garrison_power': garrison, 'minimum_attackers': threshold})
print(json.dumps(objective_matrix, indent=2))

[
  {
    "target": "Owned Level 50 stronghold without defense objective",
    "defense_bonus_percent": 0,
    "wall_power": 656594,
    "garrison_power": 100000000,
    "minimum_attackers": 31455186
  },
  {
    "target": "Owned Ironwatch",
    "defense_bonus_percent": 8,
    "wall_power": 709121,
    "garrison_power": 108000000,
    "minimum_attackers": 33971601
  },
  {
    "target": "Owned Crown Citadel",
    "defense_bonus_percent": 10,
    "wall_power": 5775378,
    "garrison_power": 165000000,
    "minimum_attackers": 53367306
  }
]


In [7]:
adjustment_matrix = []
for level in [100, 125, 150, 175, 200]:
    current_wall = full_wall(level)
    adjusted_wall = full_wall(level, soft_cap=True)
    garrison = garrison_power(level, 1_000_000)
    current_threshold = minimum_attackers_for_power(current_wall + garrison)
    adjusted_threshold = minimum_attackers_for_power(adjusted_wall + garrison)
    adjustment_matrix.append({
        'level': level, 'current_wall': current_wall, 'soft_capped_wall': adjusted_wall,
        'wall_reduction': 1 - adjusted_wall / current_wall,
        'current_capture_ratio': current_threshold / 1_000_000,
        'soft_capped_capture_ratio': adjusted_threshold / 1_000_000,
    })
print(json.dumps(adjustment_matrix, indent=2))

[
  {
    "level": 100,
    "current_wall": 5250344,
    "soft_capped_wall": 5250344,
    "wall_reduction": 0.0,
    "current_capture_ratio": 2.578233,
    "soft_capped_capture_ratio": 2.578233
  },
  {
    "level": 125,
    "current_wall": 10254251,
    "soft_capped_wall": 9187844,
    "wall_reduction": 0.10399657663928841,
    "current_capture_ratio": 4.298204,
    "soft_capped_capture_ratio": 3.964952
  },
  {
    "level": 150,
    "current_wall": 17719094,
    "soft_capped_wall": 13125344,
    "wall_reduction": 0.2592542259779196,
    "current_capture_ratio": 6.787218,
    "soft_capped_capture_ratio": 5.351671
  },
  {
    "level": 175,
    "current_wall": 28137063,
    "soft_capped_wall": 17062844,
    "wall_reduction": 0.3935811992886393,
    "current_capture_ratio": 10.199083,
    "soft_capped_capture_ratio": 6.73839
  },
  {
    "level": 200,
    "current_wall": 42000344,
    "soft_capped_wall": 21000344,
    "wall_reduction": 0.4999959047954464,
    "current_capture_ratio": 14

## Results

The current model meets the intended Level 50/75/100 benchmarks and makes coordinated waves valuable. Two mathematical effects are not healthy: capture-threshold survivor counts jump discontinuously from zero to roughly 32% of the attacking army, and uncapped cubic walls push the fresh capture ratio from 2.58× at Level 100 to 14.69× at Level 200.

Objective bonuses are bounded: Ironwatch raises its maximum-fortification one-wave threshold by 8%, and the Citadel raises its threshold by 10%. These bonuses affect defense only; march-speed bonuses change timing but not attack power.

## Takeaways

1. Replace the winning-survivor calculation with remaining attack power after 100% of resolved defense, eliminating the one-troop cliff in survivor counts.
2. Keep the current wall formula through Level 100, then continue linearly from the Level 100 slope. This preserves all approved benchmarks while reducing the Level 200 one-wave ratio from 14.69× to about 8.13× against 1 million defenders.
3. Keep the 5% persistence threshold, 30-minute repair timer, intact-wall 10% defender-loss cap, and objective bonus percentages for the first release.
4. Add production telemetry for wall integrity before combat, wall layer result, garrison layer result, city level, skill bonuses, attack/defense powers, losses, capture, and time to follow-up. Live data is required to tune the repair window confidently.